# CMF Unlearning — Kaggle Notebook

**Class-Mean-Feature (CMF) Machine Unlearning** — a framework for selectively forgetting classes  
from a trained neural network without full retraining.

This notebook walks through the full pipeline:
1. Environment setup & repo clone
2. Pre-train a model (CIFAR-10 / ResNet-18)
3. Run a baseline unlearning method (retrain from scratch on retain set)
4. Run CMF unlearning (`grad_ascent_descent_CMF_RemoveFC`)
5. Compare accuracy on retain vs. forget classes

> **GPU recommended.** Enable *GPU T4 x2* or *P100* in **Settings → Accelerator**.

## 1. Environment Setup

In [ ]:
import subprocess, sys

def run(cmd):
    """Run a shell command and stream output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-3000:])   # tail to avoid flooding
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
    return result.returncode

# Install / upgrade required packages
run("pip install -q timm einops scikit-learn matplotlib seaborn")

In [ ]:
import os, sys, json, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the Repository

In [ ]:
REPO_DIR = "/kaggle/working/CMF_Unlearning"

if not os.path.isdir(REPO_DIR):
    run(f"git clone https://github.com/your-username/CMF_Unlearning-main.git {REPO_DIR}")
    # ↑ Replace with your actual repo URL.
    #   Alternatively, upload the project as a Kaggle Dataset and
    #   set REPO_DIR = "/kaggle/input/cmf-unlearning"
else:
    print("Repo already present.")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working directory:", os.getcwd())

## 3. Configuration

Edit the variables below to switch dataset, architecture, or forget classes.

In [ ]:
# ── Core experiment config ──────────────────────────────────────────────
DATASET        = "cifar10"       # "cifar10" | "cifar100" | "tinyimagenet"
ARCH           = "resnet18"      # "resnet18" | "resnet50" | "vit_s_16"
FORGET_CLASSES = [0]             # list of class indices to forget
SEED           = 1234
DATA_PATH      = "/kaggle/working/data"

# ── Training hyper-parameters ──────────────────────────────────────────
PRETRAIN_EPOCHS  = 100           # reduce for a quick demo (e.g., 20)
PRETRAIN_LR      = 0.1
PRETRAIN_BS      = 128
UNLEARN_EPOCHS   = 3
UNLEARN_LR       = 1e-4
UNLEARN_BS       = 128

# ── Dataset size constants ─────────────────────────────────────────────
_TOTAL     = {"cifar10": 50000, "cifar100": 50000, "tinyimagenet": 100000}
_PER_CLASS = {"cifar10": 5000,  "cifar100": 500,   "tinyimagenet": 500}

n_forget       = len(FORGET_CLASSES)
NUM_FORGET     = n_forget * _PER_CLASS[DATASET]
NUM_RETAIN     = _TOTAL[DATASET] - NUM_FORGET
FORGET_STR     = ",".join(str(c) for c in FORGET_CLASSES)

print(f"Dataset: {DATASET}  Arch: {ARCH}")
print(f"Forget classes: {FORGET_CLASSES}")
print(f"Retain samples: {NUM_RETAIN}  |  Forget samples: {NUM_FORGET}")

## 4. Helper — build argparse `args` from a dict

The project's `main.py` uses `argparse`. We replicate those defaults here so we can call
the library functions directly inside the notebook.

In [ ]:
import argparse

def make_args(**overrides):
    """Return a Namespace with all project defaults, overridden by `overrides`."""
    defaults = dict(
        # dataset / model
        dataset=DATASET,
        arch=ARCH,
        data_path=DATA_PATH,
        num_classes=None,          # filled by get_dataset
        class_label_names=None,    # filled by get_dataset
        # training
        batch_size=PRETRAIN_BS,
        test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR,
        momentum=0.9,
        weight_decay=5e-4,
        gamma=0.5,
        seed=SEED,
        log_interval=100,
        val_ratio=0.1,
        patience=25,
        warmup_epochs=5,
        min_lr=1e-5,
        lr_scheduler="cosine",
        # unlearning
        unlearn_method="pre_train",
        unlearn_class=list(FORGET_CLASSES),
        num_retain_samples=NUM_RETAIN,
        num_forget_samples=NUM_FORGET,
        grad_norm_clip=None,
        salun_threshold=0.1,
        goel_exact=False,
        ssd_lambda=1,
        ssd_alpha=10,
        scrub_del_bsz=512,
        scrub_sgda_bsz=64,
        scrub_msteps=2,
        scrub_epochs=3,
        SVD_alpha_r=100,
        SVD_alpha_f=3,
        SVD_samples=900,
        SVD_max_patches=10000,
        tarun_impair_lr=2e-4,
        tarun_samples_per_class=1000,
        # misc flags
        no_cuda=False,
        no_mps=True,
        dry_run=False,
        save_model=True,
        save_path=None,
        sub_set_mode=False,
        sub_set_samples=10000,
        no_train_transform=False,
        train_transform=True,
        gpu_id=0,
        multiclass=False,
        class_names=None,
        do_mia=False,
        do_mia_ulira=False,
        plot_mia_roc=False,
        prob_batch_size=64,
        freeze_except_last=False,
        zero_last_layer=False,
        remove_FC=False,
        CMF_momentum=0.9,
        CMFClassifier=True,
        do_lp=False,
        lp_every=0,
        ncc_every=0,
        eval_every_iter=0,
        lp_every_iter=0,
        ncc_every_iter=0,
        pretrained=False,
        project_name="kaggle",
        group_name="demo",
    )
    defaults.update(overrides)
    return argparse.Namespace(**defaults)

print("make_args helper ready.")

## 5. Load Dataset

In [ ]:
from utils import get_dataset, get_retain_forget_partition

os.makedirs(DATA_PATH, exist_ok=True)

args_data = make_args()
dataset_train, dataset_test = get_dataset(args_data)

# After get_dataset, num_classes and class_label_names are set on args_data
NUM_CLASSES       = args_data.num_classes
CLASS_LABEL_NAMES = args_data.class_label_names

print(f"Train size: {len(dataset_train)}  |  Test size: {len(dataset_test)}")
print(f"Classes ({NUM_CLASSES}): {CLASS_LABEL_NAMES}")

In [ ]:
retain_dataset, forget_dataset = get_retain_forget_partition(
    args_data, dataset_train, FORGET_CLASSES
)
print(f"Retain: {len(retain_dataset)}  |  Forget: {len(forget_dataset)}")

loader_kwargs = dict(batch_size=PRETRAIN_BS, num_workers=2,
                     pin_memory=True, shuffle=True)
test_kwargs   = dict(batch_size=256, num_workers=2,
                     pin_memory=True, shuffle=False)

train_loader   = torch.utils.data.DataLoader(dataset_train, **loader_kwargs)
test_loader    = torch.utils.data.DataLoader(dataset_test,  **test_kwargs)
retain_loader  = torch.utils.data.DataLoader(retain_dataset, **loader_kwargs)
forget_loader  = torch.utils.data.DataLoader(forget_dataset, **loader_kwargs)

## 6. Pre-train the Model

A pre-trained checkpoint is **required** before running any unlearning method.  
Skip this cell if you already have `checkpoints/pre_train/{dataset}_{arch}.pt`.

In [ ]:
from utils import get_model, test

CKPT_DIR    = f"{REPO_DIR}/checkpoints/pre_train"
PRETRAIN_PT = f"{CKPT_DIR}/{DATASET}_{ARCH}.pt"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

args_pretrain = make_args(
    unlearn_method="pre_train",
    epochs_or_steps=PRETRAIN_EPOCHS,
    lr=PRETRAIN_LR,
    num_classes=NUM_CLASSES,
    class_label_names=CLASS_LABEL_NAMES,
)

model = get_model(args_pretrain, device)

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR
from train import train as train_one_epoch

if os.path.exists(PRETRAIN_PT):
    print(f"Loading existing checkpoint: {PRETRAIN_PT}")
    model.load_state_dict(torch.load(PRETRAIN_PT, map_location=device))
else:
    print(f"Pre-training {ARCH} on {DATASET} for {PRETRAIN_EPOCHS} epochs ...")

    # ── Val split for early stopping ─────────────────────────────────────
    total_len = len(dataset_train)
    val_len   = int(total_len * 0.1)
    train_len = total_len - val_len
    g = torch.Generator().manual_seed(SEED)
    all_idx   = torch.randperm(total_len, generator=g).tolist()
    tr_idx, va_idx = all_idx[:train_len], all_idx[train_len:]

    tr_sub = torch.utils.data.Subset(dataset_train, tr_idx)
    va_sub = torch.utils.data.Subset(dataset_train, va_idx)
    tr_loader_pt = torch.utils.data.DataLoader(tr_sub, **loader_kwargs)
    va_loader_pt = torch.utils.data.DataLoader(va_sub, **test_kwargs)

    optimizer = optim.SGD(model.parameters(), lr=PRETRAIN_LR,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

    warmup_epochs  = 5
    cosine_epochs  = max(1, PRETRAIN_EPOCHS - warmup_epochs)
    warmup_sched   = LambdaLR(optimizer,
                              lr_lambda=lambda e: min(1.0, (e+1)/warmup_epochs))
    cosine_sched   = CosineAnnealingLR(optimizer, T_max=cosine_epochs, eta_min=1e-5)
    scheduler      = SequentialLR(optimizer,
                                  schedulers=[warmup_sched, cosine_sched],
                                  milestones=[warmup_epochs])

    best_val_acc, best_epoch, patience_count = 0.0, 0, 0
    PATIENCE = 25

    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            args_pretrain, model, device, tr_loader_pt, optimizer, epoch
        )
        val_retain_acc, val_forget_acc, _ = test(
            model, device, va_loader_pt,
            FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
            plot_cm=False, job_name="pre_train", set_name="Val"
        )
        scheduler.step()

        if val_retain_acc > best_val_acc:
            best_val_acc = val_retain_acc
            best_epoch   = epoch
            patience_count = 0
            torch.save(model.state_dict(), PRETRAIN_PT)
            print(f"  ✓ epoch {epoch}: val_acc={val_retain_acc:.4f} — saved")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break

    print(f"\nBest epoch: {best_epoch}  best_val_acc: {best_val_acc:.4f}")
    model.load_state_dict(torch.load(PRETRAIN_PT, map_location=device))

# ── Final test accuracy ───────────────────────────────────────────────
print("\n── Pre-trained model test results ──")
retain_acc, forget_acc, _ = test(
    model, device, test_loader,
    FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
    plot_cm=False, job_name="pre_train", set_name="Test"
)

## 7. Baseline Unlearning — Retrain from Scratch on Retain Set

"Gold standard" baseline: train a brand-new model using only the **retain** set.  
Expensive but gives the ideal forget–accuracy trade-off.

In [ ]:
from unlearn import unlear_func

RETRAIN_PT = f"{REPO_DIR}/checkpoints/retrain/{DATASET}_{ARCH}/{FORGET_STR}.pt"
os.makedirs(os.path.dirname(RETRAIN_PT), exist_ok=True)

args_retrain = make_args(
    unlearn_method="retrain",
    epochs_or_steps=UNLEARN_EPOCHS,
    lr=1e-2,
    num_classes=NUM_CLASSES,
    class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=NUM_RETAIN,
    num_forget_samples=NUM_FORGET,
    save_model=True,
)

# Retrain starts from a freshly initialised model (no pretrained weights)
retrain_model = get_model(args_retrain, device)
optimizer_rt  = optim.SGD(retrain_model.parameters(), lr=1e-2,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

print("Running retrain baseline ...")
retrain_model = unlear_func["retrain"](
    args=args_retrain,
    model=retrain_model,
    device=device,
    retain_loader=retain_loader,
    forget_loader=forget_loader,
    train_loader=train_loader,
    val_loader=None,
    test_loader=test_loader,
    optimizer=optimizer_rt,
    epochs=UNLEARN_EPOCHS,
    train_dataset=dataset_train,
    val_index=np.arange(len(dataset_train)),
    test_forget_loader=forget_loader,
)

torch.save(retrain_model.state_dict(), RETRAIN_PT)
print("\n── Retrain (baseline) test results ──")
rt_retain_acc, rt_forget_acc, _ = test(
    retrain_model, device, test_loader,
    FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
    plot_cm=False, job_name="retrain", set_name="Test"
)

## 8. CMF Unlearning

The CMF approach (`grad_ascent_descent_CMF_RemoveFC`) builds a **Class-Mean-Feature**  
classifier head, removes the FC layer, and applies gradient ascent/descent  
to erase the forget-class knowledge while preserving retain-class performance.

### 8a. Fine-tune the pre-trained encoder with CMF head (`CMF_FT_RemoveFC`)

In [ ]:
from utils import load_encoder_ckpt_safely

CMF_FT_PT = f"{REPO_DIR}/checkpoints/CMF_FT_RemoveFC/{DATASET}_{ARCH}.pt"
os.makedirs(os.path.dirname(CMF_FT_PT), exist_ok=True)

args_cmf_ft = make_args(
    unlearn_method="CMF_FT_RemoveFC",
    epochs_or_steps=UNLEARN_EPOCHS,
    lr=1e-3,
    num_classes=NUM_CLASSES,
    class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=NUM_RETAIN,
    num_forget_samples=NUM_FORGET,
    remove_FC=True,
    CMFClassifier=True,
    pretrained=False,
    save_model=True,
)

cmf_model = get_model(args_cmf_ft, device)

# Load weights from the pre-trained checkpoint into the CMF model encoder
print("Loading pre-trained weights into CMF model ...")
load_encoder_ckpt_safely(cmf_model, PRETRAIN_PT, device=str(device))

test(
    cmf_model, device, test_loader,
    FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
    plot_cm=False, job_name="CMF_FT_RemoveFC", set_name="Before CMF-FT"
)

optimizer_cmf_ft = optim.SGD(
    cmf_model.parameters(), lr=1e-3,
    momentum=0.9, weight_decay=5e-4, nesterov=True
)

# Build retain / forget loaders for the test set (needed by CMF)
_, test_forget_dataset = get_retain_forget_partition(
    args_cmf_ft, dataset_test, FORGET_CLASSES
)
test_forget_loader = torch.utils.data.DataLoader(test_forget_dataset, batch_size=256)

print("Running CMF_FT_RemoveFC ...")
cmf_ft_model = unlear_func["CMF_FT_RemoveFC"](
    args=args_cmf_ft,
    model=cmf_model,
    device=device,
    retain_loader=retain_loader,
    forget_loader=forget_loader,
    train_loader=train_loader,
    val_loader=None,
    test_loader=test_loader,
    optimizer=optimizer_cmf_ft,
    epochs=UNLEARN_EPOCHS,
    train_dataset=dataset_train,
    val_index=np.arange(len(dataset_train)),
    test_forget_loader=test_forget_loader,
)

torch.save(cmf_ft_model.state_dict(), CMF_FT_PT)
print("CMF_FT_RemoveFC checkpoint saved:", CMF_FT_PT)

### 8b. CMF Unlearning — `grad_ascent_descent_CMF_RemoveFC`

In [ ]:
UNLEARN_METHOD = "grad_ascent_descent_CMF_RemoveFC"
CMF_UL_PT      = (f"{REPO_DIR}/checkpoints/{UNLEARN_METHOD}/"
                  f"{DATASET}_{ARCH}/{FORGET_STR}.pt")
os.makedirs(os.path.dirname(CMF_UL_PT), exist_ok=True)

args_cmf_ul = make_args(
    unlearn_method=UNLEARN_METHOD,
    epochs_or_steps=UNLEARN_EPOCHS,
    lr=UNLEARN_LR,
    num_classes=NUM_CLASSES,
    class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=NUM_RETAIN,
    num_forget_samples=NUM_FORGET,
    remove_FC=True,
    CMFClassifier=True,
    grad_norm_clip=1.0,
    save_model=True,
)

# Fresh CMF model — load the CMF_FT checkpoint
ul_model = get_model(args_cmf_ul, device)
state = torch.load(CMF_FT_PT, map_location=device)
ul_model.load_state_dict(state)

test(
    ul_model, device, test_loader,
    FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
    plot_cm=False, job_name=UNLEARN_METHOD, set_name="Before Unlearning"
)

optimizer_ul = optim.SGD(
    ul_model.parameters(), lr=UNLEARN_LR,
    momentum=0.9, weight_decay=5e-4, nesterov=True
)

print(f"Running {UNLEARN_METHOD} ...")
unlearnt_model = unlear_func[UNLEARN_METHOD](
    args=args_cmf_ul,
    model=ul_model,
    device=device,
    retain_loader=retain_loader,
    forget_loader=forget_loader,
    train_loader=train_loader,
    val_loader=None,
    test_loader=test_loader,
    optimizer=optimizer_ul,
    epochs=UNLEARN_EPOCHS,
    train_dataset=dataset_train,
    val_index=np.arange(len(dataset_train)),
    test_forget_loader=test_forget_loader,
)

torch.save(unlearnt_model.state_dict(), CMF_UL_PT)
print("Unlearnt model saved:", CMF_UL_PT)

## 9. Results Comparison

In [ ]:
print("=" * 60)
print(" RESULTS SUMMARY")
print("=" * 60)

models_to_eval = [
    ("Pre-trained",        model),
    ("Retrain (baseline)", retrain_model),
    (f"CMF ({UNLEARN_METHOD})", unlearnt_model),
]

results = []
for name, m in models_to_eval:
    m.eval()
    ra, fa, _ = test(
        m, device, test_loader,
        FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
        plot_cm=False, job_name=name, set_name="Test"
    )
    results.append({"Model": name, "Retain Acc": ra, "Forget Acc": fa})

import pandas as pd
df = pd.DataFrame(results)
pd.set_option("display.float_format", "{:.4f}".format)
print(df.to_string(index=False))

## 10. Visualisation — Retain vs Forget Accuracy

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 120})

model_names = df["Model"].tolist()
retain_accs = df["Retain Acc"].tolist()
forget_accs = df["Forget Acc"].tolist()

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, retain_accs, width, label="Retain Acc",
               color="steelblue", alpha=0.85)
bars2 = ax.bar(x + width/2, forget_accs, width, label="Forget Acc",
               color="tomato",    alpha=0.85)

ax.set_ylabel("Accuracy")
ax.set_title(f"Retain vs Forget Accuracy — {DATASET} / {ARCH}\n"
             f"Forget classes: {FORGET_CLASSES}")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10, ha="right")
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt="%.3f", padding=3, fontsize=8)
ax.bar_label(bars2, fmt="%.3f", padding=3, fontsize=8)
fig.tight_layout()
plt.savefig(f"{REPO_DIR}/results_comparison.png")
plt.show()
print("Plot saved to results_comparison.png")

## 11. (Optional) Run Other Unlearning Methods

All methods share the same call signature. Just change `UNLEARN_METHOD` and the
corresponding hyper-parameters.

In [ ]:
# Available methods (requires pre-trained checkpoint in checkpoints/pre_train/):
#
#   Standard baselines:
#     "random_label"             — relabel forget set with random labels
#     "salun"                    — SalUn saliency-based unlearning
#     "grad_ascent_descent"      — gradient ascent on forget + descent on retain
#     "scrub"                    — SCRUB (knowledge distillation approach)
#     "tarun"                    — UNSIR (impair + repair)
#     "ssd"                      — SSD selective synaptic dampening
#     "SVD"                      — SVD-based unlearning
#
#   CMF variants (require CMF_FT / CMF_FT_RemoveFC checkpoint first):
#     "random_label_CMF_RemoveFC"
#     "salun_CMF_RemoveFC"
#     "grad_ascent_descent_CMF_RemoveFC"   ← demonstrated above
#     "tarun_CMF_RemoveFC"
#     "scrub_CMF_RemoveFC"

OTHER_METHOD = "random_label"      # ← change this
OTHER_LR     = 1e-2
OTHER_EPOCHS = 3

args_other = make_args(
    unlearn_method=OTHER_METHOD,
    epochs_or_steps=OTHER_EPOCHS,
    lr=OTHER_LR,
    num_classes=NUM_CLASSES,
    class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=NUM_RETAIN,
    num_forget_samples=NUM_FORGET,
    grad_norm_clip=1.0,
)

other_model = get_model(args_other, device)

# Load pre-trained weights
other_model.load_state_dict(torch.load(PRETRAIN_PT, map_location=device))

optimizer_other = optim.SGD(
    other_model.parameters(), lr=OTHER_LR,
    momentum=0.9, weight_decay=5e-4, nesterov=True
)

print(f"Running {OTHER_METHOD} ...")
other_unlearnt = unlear_func[OTHER_METHOD](
    args=args_other,
    model=other_model,
    device=device,
    retain_loader=retain_loader,
    forget_loader=forget_loader,
    train_loader=train_loader,
    val_loader=None,
    test_loader=test_loader,
    optimizer=optimizer_other,
    epochs=OTHER_EPOCHS,
    train_dataset=dataset_train,
    val_index=np.arange(len(dataset_train)),
    test_forget_loader=test_forget_loader,
)

print(f"\n── {OTHER_METHOD} results ──")
test(
    other_unlearnt, device, test_loader,
    FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
    plot_cm=False, job_name=OTHER_METHOD, set_name="Test"
)

## 12. (Optional) Run the Original Shell Scripts via `subprocess`

If you prefer to run the original bash scripts from the repo, you can call them  
directly. This replicates the multi-GPU pipeline on a Kaggle single-GPU environment  
by running jobs sequentially.

In [ ]:
# Example: run the pre-train script (trains all combinations listed in the script)
# Uncomment and adjust DATA_ROOT before running.

# run(f"bash {REPO_DIR}/script/run/run_resnet_unlearning.sh {DATA_PATH}")

# Example: run the CMF unlearning script
# run(f"bash {REPO_DIR}/script/run/run_resnet_18_CMF_unlearning.sh {DATA_PATH}")

# Example: run evaluation
# run(f"bash {REPO_DIR}/script/eval/eval_resnet_CMF_unlearning.sh")

print("Uncomment a line above to run a shell script.")

## 13. Quick CLI Demo (single command)

Run `main.py` exactly as the shell scripts do, but for a single experiment.

In [ ]:
cli_cmd = f"""
python {REPO_DIR}/main.py \\
  --dataset            {DATASET} \\
  --data-path          {DATA_PATH} \\
  --arch               {ARCH} \\
  --unlearn-method     grad_ascent_descent_CMF_RemoveFC \\
  --epochs-or-steps    {UNLEARN_EPOCHS} \\
  --batch-size         {UNLEARN_BS} \\
  --lr                 {UNLEARN_LR} \\
  --num-retain-samples {NUM_RETAIN} \\
  --num-forget-samples {NUM_FORGET} \\
  --unlearn-class      {FORGET_STR} \\
  --gpu-id             0 \\
  --grad-norm-clip     1.0 \\
  --remove_FC \\
  --CMFClassifier \\
  --lp_every           0 \\
  --ncc_every          0 \\
  --save-model
""".strip()

print("Command to run:\n")
print(cli_cmd)

# Uncomment to execute:
# run(cli_cmd)

---
## Summary

| Step | What happened |
|------|---------------|
| Pre-train | Full model trained on all classes → saved to `checkpoints/pre_train/` |
| Retrain baseline | New model trained **only on retain classes** — gold-standard upper bound |
| CMF_FT_RemoveFC | Pre-trained encoder fine-tuned with CMF head (FC removed) |
| grad_ascent_descent_CMF_RemoveFC | CMF unlearning — forget classes erased while retain accuracy preserved |

**Ideal outcome:**  
- *Retain accuracy* stays close to the pre-trained model.  
- *Forget accuracy* drops toward the retrain baseline (random-chance level).  

See `script/eval/` for full evaluation scripts (linear probe, NCC, MIA, t-SNE).